In [ ]:
# Imports
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torchmetrics
import torch.nn.functional as F

# Import everything from nlp_utils package
from nlp_utils import (
    # Tokenization & Data
    tokenize,
    Vocabulary,
    ReviewDataSet,
    collate_fn,
    extract_imdb_sample_as_dict,
    
    # Models
    SentimentModel,
    MaskedMeanPool,
    
    # Attention components
    scaled_dot_product_attention,
    create_padding_mask,
    MultiHeadAttention,
    PositionalEncoding,
    FeedForward,
    TransformerEncoderBlock,
    
    # Classifiers
    MHAMeanPooledClassifier,
    TransformerClassifier,
    
    # Training
    Trainer
)

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# MHAMeanPooledClassifier is now imported from nlp_utils

In [3]:
#

In [ ]:
# TransformerClassifier is now imported from nlp_utils

In [5]:
#Test TransformerClassifier - Verify Dimensions
print("Testing TransformerClassifier dimensions...")
print("=" * 60)

# Configuration
vocab_size = 5000
embedding_dim = 64
attention_heads = 4
num_target_classes = 2
batch_size = 4
seq_len = 20

# Create model
model = TransformerClassifier(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    attention_heads=attention_heads,
    num_target_classes=num_target_classes,
    max_input_len=512
)

# Move to GPU and eval mode
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device).eval()
print(f"Device: {device}\n")

# Test input
input_ids = torch.randint(0, vocab_size, (batch_size, seq_len)).to(device)
mask = torch.ones(batch_size, seq_len).to(device)
mask[:, 15:] = 0  # Last 5 positions are padding

# Forward pass
with torch.no_grad():
    logits = model(input_ids, mask=mask)

# Verify dimensions
print(f"Input dimensions:")
print(f"  input_ids: {input_ids.shape} (batch_size, seq_len)")
print(f"  mask: {mask.shape} (batch_size, seq_len)")
print()
print(f"Output dimensions:")
print(f"  logits: {logits.shape} (batch_size, num_classes)")
print()

# Assertions
assert logits.shape == (batch_size, num_target_classes), \
    f"Expected {(batch_size, num_target_classes)}, got {logits.shape}"
print(f"✅ Output shape correct: ({batch_size}, {num_target_classes})")

# Verify predictions work
predictions = logits.argmax(dim=-1)
probabilities = torch.softmax(logits, dim=-1)
assert predictions.shape == (batch_size,), f"Predictions shape wrong: {predictions.shape}"
assert probabilities.shape == (batch_size, num_target_classes), f"Probabilities shape wrong: {probabilities.shape}"
print(f"✅ Predictions shape correct: ({batch_size},)")
print(f"✅ Probabilities shape correct: ({batch_size}, {num_target_classes})")

print()
print("=" * 60)
print("✅ All dimension tests passed!")
print("=" * 60)

Testing TransformerClassifier dimensions...
Device: cuda

Input dimensions:
  input_ids: torch.Size([4, 20]) (batch_size, seq_len)
  mask: torch.Size([4, 20]) (batch_size, seq_len)

Output dimensions:
  logits: torch.Size([4, 2]) (batch_size, num_classes)

✅ Output shape correct: (4, 2)
✅ Predictions shape correct: (4,)
✅ Probabilities shape correct: (4, 2)

✅ All dimension tests passed!
